# How to run Bayesian LM specification tests

You have a spatial dataset and need to decide which model to fit: is the
dependence in the outcome (a lag), in the errors, in the neighbours'
covariates, or some combination? The Lagrange-Multiplier tests answer that
before you commit to a specification.

`neighbayes` evaluates each LM statistic at every posterior draw, so you get a
posterior distribution of the statistic rather than a single point. The full
inventory of tests, their null hypotheses and degrees of freedom, and the
Neyman-orthogonal construction behind the robust variants are in
[Supported Models](../models.md).

In [ ]:
import warnings

import libpysal
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from neighbayes.diagnostics.lmtests import (
    bayesian_lm_error_sdm_test,
    bayesian_lm_error_test,
    bayesian_lm_lag_sdem_test,
    bayesian_lm_lag_test,
    bayesian_lm_sdm_joint_test,
    bayesian_lm_slx_error_joint_test,
    bayesian_lm_wx_sem_test,
    bayesian_lm_wx_test,
    bayesian_robust_lm_error_sdem_test,
    bayesian_robust_lm_error_test,
    bayesian_robust_lm_lag_sdm_test,
    bayesian_robust_lm_lag_test,
    bayesian_robust_lm_wx_test,
)
from neighbayes.models import OLS, SAR, SDEM, SDM, SEM, SLX

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Set up data and weights

We create a simple spatial DGP with known parameters to validate the tests. Under the null hypothesis (no spatial effects), the Bayesian LM statistics should be small with high p-values.

In [ ]:
import geopandas as gpd
from libpysal.graph import Graph
from libpysal.weights import Rook

# Generate data under H0: no spatial effects
np.random.seed(42)

# Load Columbus dataset for spatial weights
columbus_path = libpysal.examples.get_path("columbus.shp")
gdf = gpd.read_file(columbus_path)

# Create a Graph (modern libpysal API) for neighbayes models
# Row-standardize the graph so spatial models work correctly
g = Graph.build_contiguity(gdf, rook=True).transform("r")
n = g.n

# Legacy W for spreg comparison
w_spreg = Rook.from_shapefile(columbus_path)
w_spreg.transform = "r"

# Get sparse and dense W matrices
W_sparse = g.sparse.tocsr().astype(np.float64)
W_dense = np.array(W_sparse.todense())

# Design matrix
k = 3
X = np.column_stack([np.ones(n), np.random.normal(size=(n, k - 1))])
beta_true = np.array([1.0, 2.0, -1.5])
y = X @ beta_true + np.random.normal(scale=1.0, size=n)

print(f"W shape: {W_sparse.shape}, nnz: {W_sparse.nnz}")

## Fit the null models the tests need

The Bayesian LM tests require posterior draws from the **null model** (the model under H₀). Different tests use different null models:

- **LM-WX test**: SAR model (includes ρ but not γ)
- **LM-SDM joint test**: OLS model (no spatial params)
- **LM-SLX-Error joint test**: OLS model (no spatial params)
- **Robust LM-Lag-SDM**: SLX model (includes γ but not ρ)
- **Robust LM-WX**: SAR model (includes ρ but not γ)
- **Robust LM-Error-SDEM**: SLX model (includes γ but not λ)

In [ ]:
# Fit OLS model (null for joint tests)
ols_model = OLS(y=y, X=X, W=g)
ols_model.fit(draws=5000, tune=5000, chains=4, random_seed=42)

# Fit SAR model (null for LM-WX and robust LM-WX)
sar_model = SAR(y=y, X=X, W=g)
sar_model.fit(draws=5000, tune=5000, chains=4, random_seed=42)

# Fit SLX model (null for robust LM-Lag-SDM and robust LM-Error-SDEM)
slx_model = SLX(y=y, X=X, W=g)
slx_model.fit(draws=5000, tune=5000, chains=4, random_seed=42)

## Test for a lag, an error, or lagged covariates

These tests assume the nuisance parameters are correctly specified (zero). They are the Bayesian analogues of the classical LM tests from Koley & Bera (2024).

In [ ]:
# Fit the SEM null required by LM-WX-SEM (Bayesian analogue of Koley–Bera 2024).
sem_model = SEM(y=y, X=X, W=g)
sem_model.fit(draws=2500, tune=2500, chains=2, random_seed=42)

# Non-robust Bayesian LM tests evaluated at the appropriate null model.
non_robust_results = pd.DataFrame(
    [
        bayesian_lm_lag_test(ols_model).to_series(),
        bayesian_lm_error_test(ols_model).to_series(),
        bayesian_lm_wx_test(sar_model).to_series(),
        bayesian_lm_wx_sem_test(sem_model).to_series(),
        bayesian_lm_sdm_joint_test(ols_model).to_series(),
        bayesian_lm_slx_error_joint_test(ols_model).to_series(),
    ],
    index=[
        "LM-Lag",
        "LM-Error",
        "LM-WX",
        "LM-WX-SEM",
        "LM-SDM Joint",
        "LM-SLX-Error Joint",
    ],
)

non_robust_results

## Use the robust variants when both channels may be present

These tests use the **Neyman orthogonal score adjustment** from Dogan et al. (2021, Proposition 3) to ensure robustness against local misspecification in the nuisance parameter. This is the key innovation over the classical Bera-Yoon (1993) approach.

The adjustment removes the correlation between the test parameter score and the nuisance parameter score:

$$g_\psi^* = g_\psi - J_{\psi\phi \cdot \sigma} \, J_{\phi\phi \cdot \sigma}^{-1} \, g_\phi$$

where $J_{\cdot \cdot \cdot \sigma}$ denotes information matrix blocks partitioned on $\sigma^2$.

In [ ]:
# Robust Bayesian LM tests (Neyman orthogonal score)
robust_results = pd.DataFrame(
    [
        bayesian_robust_lm_lag_test(ols_model).to_series(),
        bayesian_robust_lm_error_test(ols_model).to_series(),
        bayesian_robust_lm_lag_sdm_test(slx_model).to_series(),
        bayesian_robust_lm_wx_test(sar_model).to_series(),
        bayesian_robust_lm_error_sdem_test(slx_model).to_series(),
    ],
    index=[
        "Robust LM-Lag",
        "Robust LM-Error",
        "Robust LM-Lag-SDM",
        "Robust LM-WX",
        "Robust LM-Error-SDEM",
    ],
)

robust_results

## Read the whole posterior, not just the point statistic

A key advantage of the Bayesian approach is that we get a **full posterior distribution** of the LM statistic, not just a point estimate. This allows us to compute credible intervals and posterior probabilities.

In [ ]:
from scipy import stats as sp_stats

# Show six representative posterior LM distributions (a mix of non-robust and
# robust variants).  Each panel overlays the chi-squared 95% reference and the
# posterior mean.
panels = [
    ("LM-Lag", bayesian_lm_lag_test(ols_model)),
    ("LM-Error", bayesian_lm_error_test(ols_model)),
    ("LM-WX", bayesian_lm_wx_test(sar_model)),
    ("LM-SDM Joint", bayesian_lm_sdm_joint_test(ols_model)),
    ("Robust LM-WX", bayesian_robust_lm_wx_test(sar_model)),
    ("Robust LM-Lag-SDM", bayesian_robust_lm_lag_sdm_test(slx_model)),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, res) in zip(axes.flat, panels):
    ax.hist(res.lm_samples, bins=50, density=True, alpha=0.7, color="steelblue")
    chi2_ref = sp_stats.chi2.ppf(0.95, res.df)
    ax.axvline(
        chi2_ref,
        color="red",
        linestyle="--",
        label=f"$\\chi^2_{{0.95,\\,df={res.df}}}$ = {chi2_ref:.2f}",
    )
    ax.axvline(res.mean, color="black", linestyle="-", label=f"Mean = {res.mean:.2f}")
    ax.set_title(name)
    ax.legend(fontsize=8)

plt.suptitle("Posterior Distributions of Bayesian LM Statistics", fontsize=14)
plt.tight_layout()
plt.show()

## Get a recommendation instead of a table

Every fitted spatial model exposes `spatial_diagnostics()` and `spatial_diagnostics_decision()`.
The registry on each class wires the *correct* tests for its specification:

- **OLS** → `LM-Lag`, `LM-Error`, `LM-SDM-Joint`, `LM-SLX-Error-Joint`, `Robust-LM-Lag`, `Robust-LM-Error`
- **SAR** → `LM-Error`, `LM-WX`, `Robust-LM-WX`
- **SEM** → `LM-Lag`, `LM-WX`
- **SLX** → `LM-Lag`, `LM-Error`, `Robust-LM-Lag-SDM`, `Robust-LM-Error-SDEM`
- **SDM** → `LM-Error-SDM` (uses correct $e = y - \rho Wy - X\beta - WX\gamma$ residuals)
- **SDEM** → `LM-Lag-SDEM` (uses $(I-\lambda W)$-filtered residuals)


In [ ]:
# Method-based API: one call returns all wired tests as a DataFrame
ols_model.spatial_diagnostics()

In [ ]:
# The decision routine walks the appropriate tests and returns a recommendation
print("OLS recommends:")
ols_model.spatial_diagnostics_decision()

In [ ]:
print("SAR recommends:")
sar_model.spatial_diagnostics_decision()

In [ ]:
print("SLX recommends:")
slx_model.spatial_diagnostics_decision()

In [ ]:
# SDM/SDEM-aware tests require fitted SDM / SDEM models so the residuals
# include the correct spatial filters.
sdm_model = SDM(y=y, X=X, W=g)
sdm_model.fit(draws=2000, tune=2000, chains=2, random_seed=42)

sdem_model = SDEM(y=y, X=X, W=g)
sdem_model.fit(draws=2000, tune=2000, chains=2, random_seed=42)

pd.DataFrame(
    [
        bayesian_lm_error_sdm_test(sdm_model).to_series(),
        bayesian_lm_lag_sdem_test(sdem_model).to_series(),
    ],
    index=["LM-Error-SDM", "LM-Lag-SDEM"],
)

## Run the same tests on panel data

The same Bayesian LM machinery extends to balanced spatial panels.  Tests of
the form `bayesian_panel_lm_*` and `bayesian_panel_robust_lm_*` are wired into
every panel model (`OLSPanelFE`, `SARPanelFE`, `SLXPanelFE`, …) and surfaced
through the same `spatial_diagnostics()` / `spatial_diagnostics_decision()`
methods.

Below we simulate a small balanced panel ($N = 81$, $T = 4$) from the panel
fixed-effects OLS DGP and run the full diagnostic table.

In [ ]:
from neighbayes.dgp.panel_fe import simulate_panel_sar_fe
from neighbayes.dgp.utils import rook_grid_weights
from neighbayes.models import OLSPanelFE

# 9x9 rook grid (N=81), T=4 — small enough to fit quickly but large enough for
# the LM tests to behave well.
N_panel, T_panel = 81, 4
W_panel_dense, W_panel_graph = rook_grid_weights(int(np.sqrt(N_panel)))

# Simulate from a SAR-FE DGP with moderate spatial dependence so the LM-Lag
# direction is clearly significant.
panel_sim = simulate_panel_sar_fe(
    N=N_panel,
    T=T_panel,
    rho=0.4,
    beta=np.array([1.0, 2.0]),
    W=W_panel_graph,
    seed=11,
)

panel_model = OLSPanelFE(
    y=panel_sim["y"],
    X=panel_sim["X"],
    W=W_panel_graph,
    N=N_panel,
    T=T_panel,
    effects=3,  # two-way fixed effects (unit + time)
)
panel_model.fit(draws=600, tune=600, chains=2, random_seed=11, progressbar=False)

panel_model.spatial_diagnostics()

In [ ]:
print(
    f"Panel decision tree (DGP = SAR-FE) recommends: {panel_model.spatial_diagnostics_decision(alpha=0.05, format='model')}"
)

In [ ]:
panel_model.spatial_diagnostics_decision(
    alpha=0.05,
)

## Run them on origin–destination flows

Origin–destination flow models in `neighbayes.models.flow` use Kronecker
weight matrices $W_d, W_o, W_w$ for destination, origin, and network
spillovers respectively.  The Bayesian LM family for flows tests each
direction separately (`bayesian_lm_flow_dest_test`,
`bayesian_lm_flow_orig_test`, `bayesian_lm_flow_network_test`,
`bayesian_lm_flow_intra_test`) plus a joint test
(`bayesian_lm_flow_joint_test`).  Robust variants are available for the
destination, origin, and network directions.

The registry on `OLSFlow` wires the full set; `SARFlow` (which already includes
all three lag terms) wires the robust variants for marginal-extension testing.

In [ ]:
from neighbayes.dgp.flows import generate_flow_data
from neighbayes.models import OLSFlow, SARFlow

# Simulate a small SAR flow DGP on n=8 spatial units (N = 64 OD cells).
flow_data = generate_flow_data(
    n=8,
    rho_d=0.35,
    rho_o=0.25,
    rho_w=0.10,
    beta_d=[1.0, -0.5],
    beta_o=[0.5, 0.3],
    sigma=1.0,
    seed=42,
)

y_flow = np.log(flow_data["y_vec"])  # latent SAR scale (DGP default is lognormal)
X_flow = flow_data["X"]
G_flow = flow_data["G"]

ols_flow = OLSFlow(y_flow, X_flow, G_flow, col_names=flow_data["col_names"])
ols_flow.fit(draws=600, tune=600, chains=2, random_seed=42, progressbar=False)

ols_flow.spatial_diagnostics()

In [ ]:
flow_data.keys()

In [ ]:
# Fit the SAR flow alternative; its diagnostic registry consists of the robust
# (Neyman-orthogonal) marginal-extension tests.
sar_flow = SARFlow(y_flow, X_flow, G_flow, col_names=flow_data["col_names"])
sar_flow.fit(draws=600, tune=600, chains=2, random_seed=42, progressbar=False)

sar_flow.spatial_diagnostics()

## See also

- [Supported Models](../models.md) — the full test inventory, null hypotheses,
  degrees of freedom, and the Neyman-orthogonal score
- [LM tests vs `spreg`](../validation/lm_tests_spreg_comparison.ipynb) —
  cross-implementation check against the classical statistics
- [LM decision-tree recovery](../validation/lm_tests_dgp_recovery.ipynb) —
  whether the tree lands on the model that generated the data
- [How to run spatial block cross-validation](spatial_cv_demo.ipynb) — the
  out-of-sample counterpart to specification testing